# 火灾检测流程演示 (Detect 版本)

本 Notebook 演示完整的火灾检测流程：
1. 从视频/RTSP流中读取帧
2. 使用 YOLO **Detect** 模型检测火焰/烟雾（不使用分割）
3. 使用时序分类模型判断动态/静态
4. 输出检测结果

与 001 的区别：使用 detect 模块替代 segment 模块，只使用边界框检测 + BBox EMA 平滑。

## 1. 环境设置与导入

In [ ]:
import sys
import cv2
import torch
import numpy as np
from pathlib import Path
from typing import List, Optional, Tuple
from dataclasses import dataclass
from enum import Enum
import time
import matplotlib.pyplot as plt

# 添加项目根目录到路径
project_root = Path('.').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from convlstm import create_model, heatmap_to_prob
from detect import VideoDetectProcessor  # 使用 detect 模块

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 设备: {torch.cuda.get_device_name(0)}")

## 2. 定义检测结果类型

In [ ]:
class DetectionResult(Enum):
    """检测结果类型"""
    NO_DETECTION = "no_detection"          # 未检测到火焰/烟雾
    FIRE_DETECTED = "fire_detected"        # 检测到火灾（动态）
    STATIC_SUSPECTED = "static_suspected"  # 静态疑似（可能是图片或反光）


@dataclass
class FireDetectionOutput:
    """火灾检测输出"""
    result: DetectionResult
    confidence: float
    message: str
    frame_count: int
    detections: List[dict]
    heatmap: Optional[np.ndarray] = None

## 3. 火灾检测器类 (Detect 版本)

In [ ]:
class FireDetector:
    """火灾检测器 (使用 Detect 模型)"""

    def __init__(
        self,
        detect_model_path: str,
        temporal_model_path: str,
        seq_length: int = 10,
        frame_size: Tuple[int, int] = (640, 640),
        detect_conf: float = 0.5,
        temporal_threshold: float = 0.5,
        use_ema: bool = True,
        device: str = 'auto'
    ):
        """
        初始化火灾检测器

        Args:
            detect_model_path: YOLO Detect 模型路径
            temporal_model_path: 时序分类模型路径
            seq_length: 每次处理的帧数
            frame_size: 帧尺寸 (H, W)
            detect_conf: 检测模型置信度阈值
            temporal_threshold: 时序分类阈值
            use_ema: 是否使用 BBox EMA 平滑
            device: 推理设备
        """
        self.seq_length = seq_length
        self.frame_size = frame_size
        self.temporal_threshold = temporal_threshold

        # 设置设备
        if device == 'auto':
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)

        print(f"使用设备: {self.device}")

        # 加载检测模型
        print(f"加载检测模型: {detect_model_path}")
        self.detect_processor = VideoDetectProcessor(
            model_path=detect_model_path,
            confidence_threshold=detect_conf,
            use_ema=use_ema
        )

        # 加载时序分类模型
        print(f"加载时序分类模型: {temporal_model_path}")
        self.temporal_model = create_model(temporal_model_path)
        self.temporal_model = self.temporal_model.to(self.device)
        self.temporal_model.eval()

        total_params = sum(p.numel() for p in self.temporal_model.parameters())
        print(f"时序模型参数量: {total_params:,}")

    def process_frames(self, frames: List[np.ndarray]) -> FireDetectionOutput:
        """
        处理一批帧进行火灾检测

        Args:
            frames: 帧列表，每帧为 (H, W, 3) BGR 格式

        Returns:
            FireDetectionOutput 检测结果
        """
        if len(frames) < self.seq_length:
            return FireDetectionOutput(
                result=DetectionResult.NO_DETECTION,
                confidence=0.0,
                message=f"帧数不足: {len(frames)} < {self.seq_length}",
                frame_count=len(frames),
                detections=[]
            )

        # Step 1: 调整帧尺寸
        resized_frames = []
        for frame in frames:
            if frame.shape[:2] != self.frame_size:
                resized = cv2.resize(frame, (self.frame_size[1], self.frame_size[0]),
                                     interpolation=cv2.INTER_LINEAR)
            else:
                resized = frame
            resized_frames.append(resized)

        # Step 2: 使用检测模型检测火焰/烟雾
        all_detections = []
        has_detection = False
        
        # 收集每帧的检测结果
        frame_detections = []  # 每帧的检测列表
        for processed in self.detect_processor.process_frames(
            resized_frames, skip_no_detection=False
        ):
            if processed is not None and processed.has_detections:
                has_detection = True
                frame_detections.append(processed.detections)
                all_detections.extend(processed.detections)
            else:
                frame_detections.append([])

        # Step 3: 检查是否有检测目标
        if not has_detection:
            return FireDetectionOutput(
                result=DetectionResult.NO_DETECTION,
                confidence=0.0,
                message="未检测到火焰或烟雾",
                frame_count=len(frames),
                detections=[]
            )

        # Step 4: 准备时序模型输入
        # 对于 detect 模型，我们使用检测框区域创建掩码，然后送入时序模型
        temporal_input = []
        for i, frame in enumerate(resized_frames):
            # 创建基于检测框的掩码帧
            masked_frame = self._create_masked_frame(frame, frame_detections[i])
            frame_rgb = cv2.cvtColor(masked_frame, cv2.COLOR_BGR2RGB)
            frame_chw = frame_rgb.transpose(2, 0, 1).astype(np.float32) / 255.0
            temporal_input.append(frame_chw)

        # 确保有足够的帧，不足时用空帧填充
        while len(temporal_input) < self.seq_length:
            temporal_input.append(temporal_input[-1] if temporal_input else
                                  np.zeros((3, self.frame_size[0], self.frame_size[1]),
                                           dtype=np.float32))

        # 只保留最近的 N 帧，N 为 seq_length
        temporal_input = temporal_input[-self.seq_length:]

        # Step 5: 时序分类推理
        frames_tensor = torch.from_numpy(np.stack(temporal_input, axis=0))
        frames_tensor = frames_tensor.unsqueeze(0).to(self.device)

        # 计算热力图和概率
        with torch.no_grad():
            heatmap = self.temporal_model(frames_tensor)
            prob = heatmap_to_prob(heatmap).item()

        heatmap_np = heatmap[0, 0].cpu().numpy()

        # Step 6: 判断结果
        if prob > self.temporal_threshold:
            return FireDetectionOutput(
                result=DetectionResult.FIRE_DETECTED,
                confidence=prob,
                message=f"[警告] 发现火灾! 动态特征置信度: {prob:.3f}",
                frame_count=len(frames),
                detections=all_detections,
                heatmap=heatmap_np
            )
        else:
            return FireDetectionOutput(
                result=DetectionResult.STATIC_SUSPECTED,
                confidence=prob,
                message=f"未发现火灾，疑似静态图片 (静态置信度: {1-prob:.3f})",
                frame_count=len(frames),
                detections=all_detections,
                heatmap=heatmap_np
            )

    def _create_masked_frame(self, frame: np.ndarray, detections: List[dict]) -> np.ndarray:
        """
        根据检测框创建掩码帧，只保留检测区域
        
        Args:
            frame: 原始帧 (H, W, 3)
            detections: 检测结果列表
            
        Returns:
            掩码后的帧，检测框外的区域设为黑色
        """
        if not detections:
            return np.zeros_like(frame)
        
        # 创建掩码
        mask = np.zeros(frame.shape[:2], dtype=np.uint8)
        
        for det in detections:
            bbox = det.get('bbox')
            if bbox is None:
                continue
            
            x1, y1, x2, y2 = [int(v) for v in bbox]
            # 确保坐标有效
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(frame.shape[1], x2)
            y2 = min(frame.shape[0], y2)
            
            # 在掩码中标记检测区域
            mask[y1:y2, x1:x2] = 1
        
        # 应用掩码
        masked_frame = frame.copy()
        masked_frame[mask == 0] = 0
        
        return masked_frame

## 4. 视频源类

In [ ]:
class VideoSource:
    """视频源封装类，支持视频文件和 RTSP 流"""

    def __init__(self, source: str, loop: bool = False):
        """
        初始化视频源

        Args:
            source: 视频文件路径或 RTSP 流地址
            loop: 是否循环播放（仅对视频文件有效）
        """
        self.source = source
        self.loop = loop
        self.is_rtsp = source.lower().startswith('rtsp://')
        self.cap = None
        self.frame_count = 0
        self.fps = 0
        self._open()

    def _open(self):
        """打开视频源"""
        self.cap = cv2.VideoCapture(self.source)
        if not self.cap.isOpened():
            raise ValueError(f"无法打开视频源: {self.source}")

        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30
        if not self.is_rtsp:
            self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def read_frames(self, count: int) -> List[np.ndarray]:
        """读取指定数量的帧"""
        frames = []
        for _ in range(count):
            ret, frame = self.cap.read()
            if not ret:
                if self.loop and not self.is_rtsp:
                    self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    ret, frame = self.cap.read()
                    if not ret:
                        break
                else:
                    break
            frames.append(frame)
        return frames

    def skip_frames(self, count: int):
        """跳过指定数量的帧"""
        for _ in range(count):
            ret, _ = self.cap.read()
            if not ret:
                break

    def release(self):
        """释放视频源"""
        if self.cap is not None:
            self.cap.release()

    @property
    def is_opened(self) -> bool:
        return self.cap is not None and self.cap.isOpened()

## 5. 可视化函数

In [ ]:
def visualize_result(
    frame: np.ndarray,
    output: FireDetectionOutput,
    figsize: Tuple[int, int] = (12, 5),
    heatmap_threshold: float = 0.5,
    blink_frame: int = 0
):
    """
    可视化检测结果

    Args:
        frame: 原始帧 (BGR)
        output: 检测结果
        figsize: 图像尺寸
        heatmap_threshold: heatmap 高热区域阈值 (归一化后)
        blink_frame: 闪烁帧计数器，用于控制红框闪烁效果 (偶数显示，奇数隐藏)
    """
    fig, axes = plt.subplots(1, 2 if output.heatmap is not None else 1, figsize=figsize)
    
    if output.heatmap is not None:
        ax1, ax2 = axes
    else:
        ax1 = axes

    # 复制帧用于绘制
    frame_draw = frame.copy()
    frame_rgb = cv2.cvtColor(frame_draw, cv2.COLOR_BGR2RGB)
    
    # 归一化 heatmap 用于判断高热区域
    heatmap_normalized = None
    if output.heatmap is not None:
        heatmap_min = output.heatmap.min()
        heatmap_max = output.heatmap.max()
        if heatmap_max > heatmap_min:
            heatmap_normalized = (output.heatmap - heatmap_min) / (heatmap_max - heatmap_min)
        else:
            heatmap_normalized = np.zeros_like(output.heatmap)
    
    # 绘制检测框
    for det in output.detections:
        bbox = det.get('bbox')  # [x1, y1, x2, y2]
        class_name = det.get('class_name', '').lower()
        
        if bbox is None:
            continue
            
        x1, y1, x2, y2 = [int(v) for v in bbox]
        
        # 根据类别选择颜色：火焰=黄色，烟雾=绿色
        if 'fire' in class_name or '火' in class_name:
            color = (255, 255, 0)  # 黄色 (RGB)
            label = "Fire"
        elif 'smoke' in class_name or '烟' in class_name:
            color = (0, 255, 0)    # 绿色 (RGB)
            label = "Smoke"
        else:
            color = (255, 165, 0)  # 橙色 (RGB)
            label = class_name
        
        # 检查是否需要用红色闪烁框标记（heatmap 高热区域）
        use_red_blink = False
        if (output.result == DetectionResult.FIRE_DETECTED and 
            heatmap_normalized is not None):
            # 计算 bbox 中心点在 heatmap 中的位置
            h_frame, w_frame = frame.shape[:2]
            h_heatmap, w_heatmap = heatmap_normalized.shape
            
            hm_x1 = int(x1 * w_heatmap / w_frame)
            hm_y1 = int(y1 * h_heatmap / h_frame)
            hm_x2 = int(x2 * w_heatmap / w_frame)
            hm_y2 = int(y2 * h_heatmap / h_frame)
            
            hm_x1 = max(0, min(hm_x1, w_heatmap - 1))
            hm_y1 = max(0, min(hm_y1, h_heatmap - 1))
            hm_x2 = max(0, min(hm_x2, w_heatmap))
            hm_y2 = max(0, min(hm_y2, h_heatmap))
            
            if hm_x2 > hm_x1 and hm_y2 > hm_y1:
                region_mean = heatmap_normalized[hm_y1:hm_y2, hm_x1:hm_x2].mean()
                if region_mean > heatmap_threshold:
                    use_red_blink = True
        
        # 绘制边框
        if use_red_blink:
            if blink_frame % 2 == 0:
                cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (255, 0, 0), 4)
                cv2.putText(frame_rgb, "FIRE!", (x1, y1 - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        else:
            cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame_rgb, label, (x1, y1 - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # 显示带标注的帧
    ax1.imshow(frame_rgb)
    
    # 设置标题颜色
    if output.result == DetectionResult.FIRE_DETECTED:
        title_color = 'red'
        status = 'FIRE DETECTED!'
    elif output.result == DetectionResult.STATIC_SUSPECTED:
        title_color = 'orange'
        status = 'Static Suspected'
    else:
        title_color = 'green'
        status = 'No Detection'
    
    ax1.set_title(f"{status}\nConfidence: {output.confidence:.3f}", 
                  color=title_color, fontsize=14, fontweight='bold')
    ax1.axis('off')

    # 显示热力图
    if output.heatmap is not None:
        im = ax2.imshow(output.heatmap, cmap='jet', interpolation='bilinear')
        ax2.set_title('Temporal Heatmap', fontsize=12)
        ax2.axis('off')
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


def show_detection_summary(stats: dict):
    """显示检测统计摘要"""
    print("\n" + "=" * 50)
    print("检测统计")
    print("=" * 50)
    print(f"  处理批次: {stats['batch_count']}")
    print(f"  火灾检出: {stats['fire_count']} 次")
    print(f"  静态疑似: {stats['static_count']} 次")
    print(f"  无检测: {stats['no_detection_count']} 次")
    print("=" * 50)

## 6. 配置参数

In [ ]:
# ============ 配置参数 ============

# 输入源 (视频文件路径或 RTSP 流地址)
VIDEO_SOURCE = "D03_20211221160820.mp4"  # 修改为你的视频路径

# 模型路径
DETECT_MODEL_PATH = "detect.pt"     # YOLO Detect 模型
TEMPORAL_MODEL_PATH = "convlstm.pth"  # 时序分类模型

# 检测参数
SEQ_LENGTH = 10          # 每次处理的帧数
FRAME_SIZE = (640, 640)  # 帧尺寸 (H, W)
DETECT_CONF = 0.5        # 检测模型置信度阈值
TEMPORAL_THRESHOLD = 0.5 # 时序分类阈值
USE_EMA = True           # 是否使用 BBox EMA 平滑

# 设备
DEVICE = 'auto'  # 'cuda', 'cpu', 或 'auto'

# 处理参数
SKIP_FRAMES = 0    # 每次检测后跳过的帧数
MAX_BATCHES = None # 最大处理批次数 (设为 None 则处理整个视频)
LOOP_VIDEO = False # 是否循环播放视频

# 输出设置
OUTPUT_VIDEO_PATH = "output.mp4"      # 输出视频路径
PREVIEW_OUTPUT_DIR = "preview_frames" # 中间结果图片保存目录
PREVIEW_INTERVAL_SEC = 1.0            # 每隔多少秒保存一张预览图

## 7. 初始化检测器

In [ ]:
print("=" * 50)
print("火灾检测系统初始化 (Detect 版本)")
print("=" * 50)

detector = FireDetector(
    detect_model_path=DETECT_MODEL_PATH,
    temporal_model_path=TEMPORAL_MODEL_PATH,
    seq_length=SEQ_LENGTH,
    frame_size=FRAME_SIZE,
    detect_conf=DETECT_CONF,
    temporal_threshold=TEMPORAL_THRESHOLD,
    use_ema=USE_EMA,
    device=DEVICE
)

print("\n初始化完成!")

## 8. 运行检测

In [ ]:
# 打开视频源
print(f"打开视频源: {VIDEO_SOURCE}")
video_source = VideoSource(VIDEO_SOURCE, loop=LOOP_VIDEO)
print(f"视频 FPS: {video_source.fps:.1f}")
if video_source.frame_count > 0:
    print(f"视频总帧数: {video_source.frame_count}")
    total_duration = video_source.frame_count / video_source.fps
    print(f"视频时长: {total_duration:.1f}秒")

# 输出视频设置
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = None  # 延迟初始化，等获取第一帧尺寸

# 创建预览图片输出目录
preview_dir = Path(PREVIEW_OUTPUT_DIR)
preview_dir.mkdir(exist_ok=True)
print(f"预览图片保存目录: {preview_dir.absolute()}")

# 计算预览采样间隔 (每隔多少帧保存一张图)
preview_frame_interval = int(video_source.fps * PREVIEW_INTERVAL_SEC)
print(f"预览采样: 每 {PREVIEW_INTERVAL_SEC} 秒 ({preview_frame_interval} 帧) 保存一张图")

# 统计变量
stats = {
    'batch_count': 0,
    'fire_count': 0,
    'static_count': 0,
    'no_detection_count': 0
}

def draw_result_on_frame(frame: np.ndarray, output: FireDetectionOutput, 
                         frame_idx: int = 0, video_time: float = 0.0,
                         heatmap_threshold: float = 0.5) -> np.ndarray:
    """在帧上绘制检测结果，返回带标注的帧"""
    frame_draw = frame.copy()
    h_frame, w_frame = frame.shape[:2]
    
    # 模型推理尺寸 (bbox 坐标基于此尺寸)
    model_size = FRAME_SIZE[0]  # 640
    
    # 计算缩放比例
    scale_x = w_frame / model_size
    scale_y = h_frame / model_size
    
    # 归一化 heatmap
    heatmap_normalized = None
    if output.heatmap is not None:
        heatmap_min = output.heatmap.min()
        heatmap_max = output.heatmap.max()
        if heatmap_max > heatmap_min:
            heatmap_normalized = (output.heatmap - heatmap_min) / (heatmap_max - heatmap_min)
        else:
            heatmap_normalized = np.zeros_like(output.heatmap)
    
    # 绘制检测框
    for det in output.detections:
        bbox = det.get('bbox')
        class_name = det.get('class_name', '').lower()
        
        if bbox is None:
            continue
        
        # 将 bbox 从模型尺寸 (640x640) 缩放到原始帧尺寸
        x1 = int(bbox[0] * scale_x)
        y1 = int(bbox[1] * scale_y)
        x2 = int(bbox[2] * scale_x)
        y2 = int(bbox[3] * scale_y)
        
        # 确保坐标在有效范围内
        x1 = max(0, min(x1, w_frame - 1))
        y1 = max(0, min(y1, h_frame - 1))
        x2 = max(0, min(x2, w_frame))
        y2 = max(0, min(y2, h_frame))
        
        # 根据类别选择颜色 (BGR)
        if 'fire' in class_name or '火' in class_name:
            color = (0, 255, 255)  # 黄色
            label = "Fire"
        elif 'smoke' in class_name or '烟' in class_name:
            color = (0, 255, 0)    # 绿色
            label = "Smoke"
        else:
            color = (0, 165, 255)  # 橙色
            label = class_name
        
        # 检查是否需要红色框 (火灾检出且在高热区域)
        use_red = False
        if output.result == DetectionResult.FIRE_DETECTED and heatmap_normalized is not None:
            h_heatmap, w_heatmap = heatmap_normalized.shape
            
            # bbox 坐标转换到 heatmap 坐标 (使用原始 bbox，因为都是基于 640x640)
            hm_x1 = int(bbox[0] * w_heatmap / model_size)
            hm_y1 = int(bbox[1] * h_heatmap / model_size)
            hm_x2 = int(bbox[2] * w_heatmap / model_size)
            hm_y2 = int(bbox[3] * h_heatmap / model_size)
            
            hm_x1 = max(0, min(hm_x1, w_heatmap - 1))
            hm_y1 = max(0, min(hm_y1, h_heatmap - 1))
            hm_x2 = max(0, min(hm_x2, w_heatmap))
            hm_y2 = max(0, min(hm_y2, h_heatmap))
            
            if hm_x2 > hm_x1 and hm_y2 > hm_y1:
                region_mean = heatmap_normalized[hm_y1:hm_y2, hm_x1:hm_x2].mean()
                if region_mean > heatmap_threshold:
                    use_red = True
        
        # 绘制边框
        if use_red:
            cv2.rectangle(frame_draw, (x1, y1), (x2, y2), (0, 0, 255), 4)
            cv2.putText(frame_draw, "FIRE!", (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        else:
            cv2.rectangle(frame_draw, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame_draw, label, (x1, y1 - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    # 添加状态文字
    if output.result == DetectionResult.FIRE_DETECTED:
        status_text = f"FIRE DETECTED! Conf: {output.confidence:.2f}"
        status_color = (0, 0, 255)  # 红色
    elif output.result == DetectionResult.STATIC_SUSPECTED:
        status_text = f"Static Suspected ({1-output.confidence:.2f})"
        status_color = (0, 165, 255)  # 橙色
    else:
        status_text = "No Detection"
        status_color = (0, 255, 0)  # 绿色
    
    cv2.putText(frame_draw, status_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 1.0, status_color, 2)
    
    # 添加时间戳信息
    time_text = f"Frame: {frame_idx} | Time: {video_time:.1f}s"
    cv2.putText(frame_draw, time_text, (10, 60),
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return frame_draw

print("\n开始检测并生成视频...")
print("-" * 50)

global_frame_idx = 0
last_preview_frame = -preview_frame_interval  # 确保第一帧会被保存
preview_count = 0

try:
    while video_source.is_opened:
        if MAX_BATCHES is not None and stats['batch_count'] >= MAX_BATCHES:
            print(f"\n已达到最大批次数: {MAX_BATCHES}")
            break

        frames = video_source.read_frames(SEQ_LENGTH)
        if len(frames) == 0:
            break

        stats['batch_count'] += 1
        start_time = time.time()

        # 执行检测
        output = detector.process_frames(frames)
        elapsed = time.time() - start_time

        # 统计
        if output.result == DetectionResult.FIRE_DETECTED:
            stats['fire_count'] += 1
        elif output.result == DetectionResult.STATIC_SUSPECTED:
            stats['static_count'] += 1
        else:
            stats['no_detection_count'] += 1

        print(f"[Batch {stats['batch_count']:04d}] {output.message} ({elapsed:.2f}s)")

        # 将每一帧写入输出视频，并按间隔保存预览图
        for i, frame in enumerate(frames):
            current_frame_idx = global_frame_idx + i
            video_time = current_frame_idx / video_source.fps
            
            # 初始化 VideoWriter
            if out_writer is None:
                h, w = frame.shape[:2]
                out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, video_source.fps, (w, h))
                print(f"输出视频: {OUTPUT_VIDEO_PATH} ({w}x{h} @ {video_source.fps:.1f}fps)")
            
            # 绘制检测结果
            annotated_frame = draw_result_on_frame(
                frame, output, 
                frame_idx=current_frame_idx, 
                video_time=video_time
            )
            out_writer.write(annotated_frame)
            
            # 按时间间隔保存预览图
            if current_frame_idx - last_preview_frame >= preview_frame_interval:
                preview_filename = preview_dir / f"frame_{current_frame_idx:06d}_{video_time:.1f}s.jpg"
                cv2.imwrite(str(preview_filename), annotated_frame)
                preview_count += 1
                last_preview_frame = current_frame_idx
        
        global_frame_idx += len(frames)

        if SKIP_FRAMES > 0:
            video_source.skip_frames(SKIP_FRAMES)
            global_frame_idx += SKIP_FRAMES

except KeyboardInterrupt:
    print("\n检测被中断")
finally:
    video_source.release()
    if out_writer is not None:
        out_writer.release()
        print(f"\n视频已保存: {OUTPUT_VIDEO_PATH}")

print(f"预览图片已保存: {preview_count} 张 -> {preview_dir.absolute()}")
show_detection_summary(stats)
print("\n检测完成!")

## 9. 单帧测试 (可选)

In [ ]:
def test_single_batch(video_path: str, start_frame: int = 0):
    """
    测试单个批次
    
    Args:
        video_path: 视频路径
        start_frame: 起始帧位置
    """
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    frames = []
    for _ in range(SEQ_LENGTH):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    
    if len(frames) < SEQ_LENGTH:
        print(f"帧数不足: {len(frames)}")
        return
    
    print(f"测试从第 {start_frame} 帧开始的 {SEQ_LENGTH} 帧...")
    output = detector.process_frames(frames)
    print(f"结果: {output.message}")
    
    visualize_result(frames[-1], output)

# 取消注释以下行进行测试
# test_single_batch(VIDEO_SOURCE, start_frame=100)